# Explorer 07 - Artist clustering

**Purpose.** Build artist feature vectors, add node2vec graph embeddings, reduce with UMAP, and cluster with HDBSCAN.

**Feeds main report section:** 5.6 Artist clustering

**Modules used:** `vinyl.clustering`, `vinyl.networks`

**Tables read:** `release_artists`, `release_styles`, `release_formats`, `release_labels`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

**Status.** `clustering.node2vec_embeddings` and `clustering.combine_features` were implemented in T10. Expected runtime on the full 2026 data: about 30-60 minutes (node2vec 10-30 min, then two UMAP runs and HDBSCAN at a few minutes each); with `REUSE_EMBEDDINGS = True` a rerun loads the saved files and skips them.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-07-clustering")
from vinyl import clustering, networks, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_styles = ingest.read_table(spark, snapshot, "release_styles")
release_genres = ingest.read_table(spark, snapshot, "release_genres")
release_formats = ingest.read_table(spark, snapshot, "release_formats")
release_labels = ingest.read_table(spark, snapshot, "release_labels")

## 1. Artist profile features (Spark)

In [ ]:
real_artists = release_artists.where(~F.col("artist_id").isin(list(networks.PLACEHOLDER_ARTIST_IDS)))
profile = clustering.artist_profile(real_artists, release_formats, release_labels, min_releases=5).cache()
print(profile.count(), "artists kept")
profile.describe().show()

## 2. Style TF-IDF (Spark)

In [ ]:
kept = real_artists.join(profile.select("artist_id"), "artist_id", "left_semi")
tfidf = clustering.style_tfidf(kept, release_styles, min_artists=20)
tfidf.orderBy(F.desc("tfidf")).show(20, truncate=False)

## 3. node2vec embeddings

In [ ]:
# Slow steps are cached under $DISCOGS_DATA_DIR/derived/<snapshot>/ (never under /mnt/data).
# Set REUSE_EMBEDDINGS = False to recompute everything.
REUSE_EMBEDDINGS = True
DERIVED = Path(config.DATA_DIR) / "derived" / snapshot
assert not str(DERIVED).startswith("/mnt/data"), "derived files must not go under /mnt/data"
DERIVED.mkdir(parents=True, exist_ok=True)

collab = networks.collaboration_edges(
    kept, min_weight=config.GRAPH_MIN_WEIGHT["collaboration"]
).toPandas()
graph_nodes = set(collab["src"]) | set(collab["dst"])
print(len(collab), "edges,", len(graph_nodes), "nodes")

n2v_file = DERIVED / "artist_n2v.parquet"
embeddings = None
if REUSE_EMBEDDINGS and n2v_file.exists():
    embeddings = pd.read_parquet(n2v_file)
    if set(embeddings["node"]) != graph_nodes:
        print("saved embeddings do not match this graph; recomputing")
        embeddings = None
    else:
        print("loaded", n2v_file)
if embeddings is None:
    with clustering.timed("node2vec"):
        embeddings = clustering.node2vec_embeddings(collab, dimensions=64)
    embeddings.to_parquet(n2v_file, engine="pyarrow", index=False)
    print("saved", n2v_file)
embeddings.shape

## 4. Combine, reduce, cluster

In [ ]:
profile_pd = profile.toPandas()
tfidf_pd = tfidf.toPandas()

files = {name: DERIVED / f"{name}.npy" for name in ("features", "artist_ids", "embedding_2d", "embedding_10d")}
cached = REUSE_EMBEDDINGS and all(f.exists() for f in files.values())
if cached:
    artist_ids = pd.Series(np.load(files["artist_ids"]), name="artist_id")
    cached = set(artist_ids) == set(profile_pd["artist_id"])
    if not cached:
        print("saved features do not match the artist profile; recomputing")
if cached:
    features, embedding_2d, embedding_10d = (np.load(files[k]) for k in ("features", "embedding_2d", "embedding_10d"))
    print("loaded features and UMAP embeddings from", DERIVED)
else:
    with clustering.timed("combine_features"):
        features, artist_ids = clustering.combine_features(profile_pd, tfidf_pd, embeddings)
    with clustering.timed("UMAP 2d"):
        embedding_2d = clustering.reduce_umap(features, n_components=2)
    with clustering.timed("UMAP 10d"):
        embedding_10d = clustering.reduce_umap(features, n_components=10)
    for name, array in (("features", features), ("artist_ids", np.asarray(artist_ids)),
                        ("embedding_2d", embedding_2d), ("embedding_10d", embedding_10d)):
        np.save(files[name], array)
    print("saved", ", ".join(f.name for f in files.values()), "to", DERIVED)
features.shape, embedding_10d.shape

In [ ]:
# Main labels. Keep 50 unless the sweep below says otherwise; do not change it automatically.
MIN_CLUSTER_SIZE = 50
with clustering.timed("HDBSCAN"):
    cluster_labels = clustering.cluster_hdbscan(embedding_10d, min_cluster_size=MIN_CLUSTER_SIZE)
n_clusters = len(set(cluster_labels) - {-1})
print(f"{n_clusters} clusters; {(cluster_labels == -1).mean():.1%} of {len(cluster_labels)} artists are noise")

### Sensitivity to `min_cluster_size`

In [ ]:
sweep = clustering.min_cluster_size_sweep(embedding_10d, sizes=(25, 50, 100, 200))
print(sweep.to_string(index=False))

## 5. Inspect the clusters

In [ ]:
fig = clustering.plot_clusters_2d(embedding_2d, cluster_labels)
print(viz.save_figure(fig, "07_umap_clusters"))
fig

In [ ]:
# Top styles, format mix, career span and example artists of each cluster.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)
summary = clustering.describe_clusters(cluster_labels, artist_ids, profile_pd, tfidf_pd)
summary.head(30)

## 6. Compare with other groupings

In [ ]:
# Leiden communities on the same collaboration edges (resolution 1.0, seed 42).
graph = networks.to_igraph(collab)
with clustering.timed("Leiden"):
    communities = networks.leiden_communities(graph, resolution=1.0, seed=42)
print("modularity:", communities.attrs["modularity"], "| communities:", communities["community"].nunique())
vs_leiden = clustering.compare_partitions(
    cluster_labels, communities["community"].to_numpy(), artist_ids, communities["node"]
)
print({k: v for k, v in vs_leiden.items() if k != "contingency"})
vs_leiden["contingency"]

In [ ]:
# Each artist's dominant genre, over the artists that are in both tables.
dominant_genre = networks.artist_dominant_tag(real_artists, release_genres, "genre").select("artist_id", "genre").toPandas()
vs_genre = clustering.compare_partitions(
    cluster_labels, dominant_genre["genre"].to_numpy(), artist_ids, dominant_genre["artist_id"]
)
print({k: v for k, v in vs_genre.items() if k != "contingency"})
vs_genre["contingency"]

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] Minimum releases per artist and the number of artists clustered.
- [ ] node2vec library and parameters.
- [ ] Feature scaling and block weights.
- [ ] UMAP and HDBSCAN parameters, number of clusters, noise share.